PyTorch Crash Course: Deep Learning in Python

https://www.youtube.com/watch?v=uq7sbUlIDR8

In [5]:
import torch

a = torch.tensor([2., 3.], requires_grad=True)
b = torch.tensor([6., 4.], requires_grad=True)

f = 3 * a ** 3 - b ** 2

In [6]:
f

tensor([-12.,  65.], grad_fn=<SubBackward0>)

In [7]:
f.backward(gradient=torch.tensor([1, 1]))

In [9]:
print(a.grad, b.grad)

tensor([36., 81.]) tensor([-12.,  -8.])


In [12]:
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [17]:
X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# as NN are scale sensitive, we should scale the data
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)
# we don't use fit_transform again, we use the parameters that learned from scaling the training dataset

In [33]:
X_train_scaled_tensor = torch.from_numpy(X_train_scaled).float()
X_test_scaled_tensor = torch.from_numpy(X_test_scaled).float()
y_train_tensor = torch.from_numpy(y_train).unsqueeze(1).float()
y_test_tensor = torch.from_numpy(y_test).unsqueeze(1).float()

# squeeze reduce dimensionality, unsqueeze add dim

In [34]:
X_train_scaled_tensor[:1]

tensor([[ 1.7201e+00,  4.9020e-01,  1.7670e+00,  1.6948e+00,  1.4723e+00,
          1.5619e+00,  2.0704e+00,  2.5750e+00,  2.6961e+00,  5.4136e-01,
          5.6332e-01, -3.4726e-01,  5.3757e-01,  6.4464e-01, -5.1470e-01,
         -5.5598e-02, -3.3805e-03,  2.9086e-01,  7.9249e-01, -1.7256e-01,
          1.8055e+00,  9.9347e-01,  1.8403e+00,  1.8051e+00,  1.1709e+00,
          1.2672e+00,  1.2691e+00,  2.3133e+00,  4.1543e+00,  1.0376e+00]])

In [35]:
y_train_tensor[:1]

tensor([[0.]])

In [44]:
train_dataset = TensorDataset(X_train_scaled_tensor, y_train_tensor)

In [37]:
X_train_scaled_tensor.shape

torch.Size([455, 30])

In [45]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

In [46]:
# Breast Cancer Network

class BCNet(nn.Module):
    def __init__(self):
        super(BCNet, self).__init__()

        # fully connected layer = dense layer
        self.fc1 = nn.Linear(in_features=30, out_features=64)
        self.fc2 = nn.Linear(in_features=64, out_features=32)
        self.fc3 = nn.Linear(in_features=32, out_features=1)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.sigmoid(self.fc3(x))

        return x


model = BCNet()


In [47]:
# BCELoss (Binary Cross-Entropy Loss) is a loss function in machine learning used to measure the error between a model's predicted probabilities and true binary labels (0 or 1).
criterion = nn.BCELoss()

optimizer = optim.Adam(params=model.parameters(), lr=0.001)

In [48]:
epochs = 20

for epoch in range(epochs):
    model.train()
    # keep track of running loss
    running_loss = 0.0

    # iterate over the data loader
    for x_batch, y_batch in train_loader:
        optimizer.zero_grad()

        preds = model(x_batch)
        loss = criterion(preds, y_batch)

        # backpropagate
        loss.backward()
        # we need to take a step with the optimizer into that direction (based on learning rate)
        optimizer.step()

        running_loss += loss.item()

    print(f"{epoch + 1} / {epochs}, Loss: {running_loss / len(train_loader)}")

1 / 20, Loss: 0.5842407643795013
2 / 20, Loss: 0.4097151517868042
3 / 20, Loss: 0.2505112459262212
4 / 20, Loss: 0.16690954367319744
5 / 20, Loss: 0.11068128719925881
6 / 20, Loss: 0.08925024544199307
7 / 20, Loss: 0.07388908465703328
8 / 20, Loss: 0.06620981618762016
9 / 20, Loss: 0.05961228373150031
10 / 20, Loss: 0.0555022061492006
11 / 20, Loss: 0.05164239825680852
12 / 20, Loss: 0.05711434005449215
13 / 20, Loss: 0.04690484280387561
14 / 20, Loss: 0.04448481186603506
15 / 20, Loss: 0.04374598190188408
16 / 20, Loss: 0.04102102434262633
17 / 20, Loss: 0.05308415163308382
18 / 20, Loss: 0.03748139367477658
19 / 20, Loss: 0.036433269983778396
20 / 20, Loss: 0.03429336917276184


In [49]:
# how can we evaluate this model on unseen data?
# with no gradient calculation,

with torch.no_grad():
    model.eval()

    preds = model(X_test_scaled_tensor)
    loss = criterion(preds, y_test_tensor).item()
    accuracy = ((preds > 0.5) == y_test_tensor).float().mean().item()

In [50]:
accuracy

0.9824561476707458